# SignBridge — Extract Sample Sign Images
Downloads the ASL datasets from Kaggle and picks **one clean image per sign** (A–Z + 0–9).
Output: `signs_samples.zip` → unzip into `assets/images/signs/` in your project.

**Setup:** Upload `kaggle.json` first.

In [ ]:
# CELL 1: Install & Setup
!pip install -q kaggle opencv-python-headless
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Done.')

In [ ]:
# CELL 2: Download Datasets
# Alphabet dataset (A-Z real photos)
!kaggle datasets download -d grassknoted/asl-alphabet -p data/ --unzip

# Digit dataset (0-9)
!kaggle datasets download -d lexset/synthetic-asl-numbers -p data/ --unzip

import os
for root, dirs, files in os.walk('data/'):
    depth = root.replace('data/', '').count(os.sep)
    if depth < 2:
        for d in sorted(dirs)[:10]:
            full = os.path.join(root, d)
            n = len([f for f in os.listdir(full) if f.lower().endswith(('.jpg','.jpeg','.png'))])
            if n > 0:
                print(f'  {full}: {n} images')

In [ ]:
# CELL 3: Find Dataset Folders
import glob

LETTERS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
DIGITS  = list('0123456789')

sign_dirs = {}

# Find alphabet folders
for letter in LETTERS:
    candidates = glob.glob(f'data/**/{letter}', recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            imgs = [f for f in os.listdir(c) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if len(imgs) > 5:
                sign_dirs[letter] = c
                break

# Find digit folders (0 = O as fallback)
for digit in DIGITS:
    search = [digit, 'O'] if digit == '0' else [digit]
    for name in search:
        candidates = glob.glob(f'data/**/{name}', recursive=True)
        for c in candidates:
            if os.path.isdir(c):
                imgs = [f for f in os.listdir(c) if f.lower().endswith(('.jpg','.jpeg','.png'))]
                if len(imgs) > 5:
                    sign_dirs[digit] = c
                    break
        if digit in sign_dirs:
            break

print(f'Found {len(sign_dirs)} / {len(LETTERS)+len(DIGITS)} sign folders')
missing = [s for s in LETTERS+DIGITS if s not in sign_dirs]
if missing:
    print(f'Missing: {missing}')
else:
    print('All signs found!')

In [ ]:
# CELL 4: Extract Best Sample Per Sign
import cv2
import numpy as np
import shutil

OUT_DIR = 'signs_samples'
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_SIZE = 400  # px square output
SKIP = 5           # skip first N images (sometimes they are test/border images)

def pick_best_image(folder, skip=5, size=400):
    """Pick one clear, well-lit image from the folder."""
    imgs = sorted([f for f in os.listdir(folder)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if not imgs:
        return None

    # Try a few candidates and pick sharpest (highest Laplacian variance)
    candidates = imgs[skip:skip+20]  # sample from middle of dataset
    if not candidates:
        candidates = imgs[:20]

    best_img = None
    best_score = -1

    for fn in candidates:
        path = os.path.join(folder, fn)
        img = cv2.imread(path)
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()  # sharpness
        if score > best_score:
            best_score = score
            best_img = img

    if best_img is None:
        return None

    # Resize to square, preserving aspect ratio with white padding
    h, w = best_img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(best_img, (nw, nh), interpolation=cv2.INTER_LANCZOS4)

    # White background pad
    canvas = np.full((size, size, 3), 248, dtype=np.uint8)
    y0 = (size - nh) // 2
    x0 = (size - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized

    return canvas

saved = []
failed = []

for sign in LETTERS + DIGITS:
    folder = sign_dirs.get(sign)
    if not folder:
        print(f'  {sign}: SKIPPED (no folder)')
        failed.append(sign)
        continue

    img = pick_best_image(folder, skip=SKIP, size=TARGET_SIZE)
    if img is None:
        print(f'  {sign}: FAILED (could not read)')
        failed.append(sign)
        continue

    out_name = f'{sign.lower()}.jpg'  # a.jpg, b.jpg ... 0.jpg, 1.jpg ...
    out_path = os.path.join(OUT_DIR, out_name)
    cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    saved.append(sign)
    print(f'  {sign} -> {out_name}')

print(f'\nSaved: {len(saved)} / {len(LETTERS)+len(DIGITS)}')
if failed:
    print(f'Failed: {failed}')

In [ ]:
# CELL 5: Preview a few samples
import matplotlib.pyplot as plt

preview_signs = ['A','B','C','L','Y','0','1','5','9']
fig, axes = plt.subplots(1, len(preview_signs), figsize=(18, 3))

for ax, sign in zip(axes, preview_signs):
    path = os.path.join(OUT_DIR, f'{sign.lower()}.jpg')
    if os.path.exists(path):
        img = cv2.imread(path)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(sign, fontsize=16, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Signs Preview', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6: Zip & Download
shutil.make_archive('signs_samples', 'zip', OUT_DIR)
size_mb = os.path.getsize('signs_samples.zip') / 1e6
print(f'signs_samples.zip: {size_mb:.1f} MB')

try:
    from google.colab import files
    files.download('signs_samples.zip')
    print('Download triggered!')
except ImportError:
    print('Find signs_samples.zip in the Files panel.')

print()
print('Next steps:')
print('  1. Unzip signs_samples.zip')
print('  2. Copy all .jpg files into: assets/images/signs/')
print('  3. Files will be named: a.jpg, b.jpg ... z.jpg, 0.jpg ... 9.jpg')